# Ch.3 — Kubernetes Basics (Azure Supplement)

> Deploy to **Azure Kubernetes Service (AKS)** — managed K8s in the cloud. This notebook requires an Azure subscription but stays within the free tier for short experiments.

**Running Example:** Deploy Flask API to AKS  
**Constraint:** Use Azure free tier (minimize costs)  
**Tech Stack:** Azure CLI, AKS, kubectl

⚠️ **Cost warning:** AKS nodes incur hourly charges (~$0.10/hour for Standard_B2s). Delete the cluster after testing.

## Prerequisites

- Azure subscription (free tier: https://azure.microsoft.com/free/)
- Azure CLI installed (`az` command)
- kubectl installed (from main notebook)

**Azure free tier includes:**
- $200 credit for 30 days
- 12 months of popular services free
- 25+ services always free

**This notebook uses:** Azure Kubernetes Service (AKS) — not part of always-free tier, but low-cost for short tests.

## Cell 1 — Azure Credentials

Set your Azure subscription ID. The pre-push hook will strip this before commits.

In [ ]:
# ── Azure Configuration ────────────────────────────────────────────────────
import subprocess
import os
import time

# ⚠️ STRIPPED BY PRE-PUSH HOOK — safe to commit
AZURE_SUBSCRIPTION_ID = ""  # Get from: az account list --output table
AZURE_RESOURCE_GROUP = "devops-demo-rg"
AZURE_LOCATION = "eastus"
AKS_CLUSTER_NAME = "devops-demo-aks"
AKS_NODE_COUNT = 2  # Minimum for production-like setup
AKS_NODE_SIZE = "Standard_B2s"  # Low-cost VM size (2 vCPU, 4GB RAM)

if not AZURE_SUBSCRIPTION_ID:
    print("⚠️  WARNING: AZURE_SUBSCRIPTION_ID is empty.")
    print("   Get your subscription ID:")
    print("   1. Run: az account list --output table")
    print("   2. Copy the 'SubscriptionId' value")
    print("   3. Set AZURE_SUBSCRIPTION_ID above")
    print("\n   This notebook will be committed with credentials stripped (safe).")
else:
    print(f"✅ Azure subscription: {AZURE_SUBSCRIPTION_ID[:8]}...")
    print(f"   Resource group: {AZURE_RESOURCE_GROUP}")
    print(f"   Location: {AZURE_LOCATION}")
    print(f"   AKS cluster: {AKS_CLUSTER_NAME}")
    print(f"   Node size: {AKS_NODE_SIZE} (x{AKS_NODE_COUNT})")

## Cell 2 — Create Azure Kubernetes Service (AKS) Cluster

Provision a managed Kubernetes cluster in Azure. This takes 5-10 minutes.

In [ ]:
# ── Create AKS Cluster ─────────────────────────────────────────────────────

if not AZURE_SUBSCRIPTION_ID:
    print("❌ Cannot proceed without AZURE_SUBSCRIPTION_ID. See Cell 1.")
else:
    # Login to Azure (opens browser)
    print("🔐 Logging into Azure...")
    subprocess.run(["az", "login"], check=False)
    
    # Set active subscription
    subprocess.run(
        ["az", "account", "set", "--subscription", AZURE_SUBSCRIPTION_ID],
        check=True
    )
    
    # Create resource group
    print(f"\n📦 Creating resource group: {AZURE_RESOURCE_GROUP}")
    result = subprocess.run(
        [
            "az", "group", "create",
            "--name", AZURE_RESOURCE_GROUP,
            "--location", AZURE_LOCATION
        ],
        capture_output=True,
        text=True
    )
    
    if result.returncode == 0:
        print(f"✅ Resource group created in {AZURE_LOCATION}")
    else:
        print(f"⚠️  Resource group may already exist: {result.stderr}")
    
    # Check if AKS cluster exists
    check_result = subprocess.run(
        [
            "az", "aks", "show",
            "--name", AKS_CLUSTER_NAME,
            "--resource-group", AZURE_RESOURCE_GROUP
        ],
        capture_output=True,
        text=True
    )
    
    if check_result.returncode == 0:
        print(f"\n✅ AKS cluster '{AKS_CLUSTER_NAME}' already exists.")
    else:
        # Create AKS cluster
        print(f"\n☁️  Creating AKS cluster: {AKS_CLUSTER_NAME}")
        print(f"   This takes 5-10 minutes. Please wait...")
        print(f"   💰 Cost estimate: ~${0.10 * AKS_NODE_COUNT:.2f}/hour")
        
        result = subprocess.run(
            [
                "az", "aks", "create",
                "--resource-group", AZURE_RESOURCE_GROUP,
                "--name", AKS_CLUSTER_NAME,
                "--node-count", str(AKS_NODE_COUNT),
                "--node-vm-size", AKS_NODE_SIZE,
                "--enable-managed-identity",
                "--generate-ssh-keys",
                "--no-wait"  # Don't block (check status manually)
            ],
            capture_output=True,
            text=True
        )
        
        if result.returncode == 0:
            print("✅ AKS cluster creation started (running in background)")
            print("\n   Check status:")
            print(f"   az aks show -n {AKS_CLUSTER_NAME} -g {AZURE_RESOURCE_GROUP} --query provisioningState -o tsv")
            print("\n   Wait for 'Succeeded' status before proceeding to Cell 3.")
        else:
            print(f"❌ Failed to create cluster: {result.stderr}")
    
    print("\n" + "="*70)
    print("⏳ Cluster creation is asynchronous. Wait 5-10 minutes, then run:")
    print(f"   az aks show -n {AKS_CLUSTER_NAME} -g {AZURE_RESOURCE_GROUP}")
    print("="*70)

## Cell 3 — Connect kubectl to AKS

Download AKS credentials and configure kubectl to use the Azure cluster.

In [ ]:
# ── Connect kubectl to AKS ─────────────────────────────────────────────────

if not AZURE_SUBSCRIPTION_ID:
    print("❌ Cannot proceed without AZURE_SUBSCRIPTION_ID. See Cell 1.")
else:
    # Check if cluster is ready
    print("🔍 Checking cluster status...")
    result = subprocess.run(
        [
            "az", "aks", "show",
            "--name", AKS_CLUSTER_NAME,
            "--resource-group", AZURE_RESOURCE_GROUP,
            "--query", "provisioningState",
            "-o", "tsv"
        ],
        capture_output=True,
        text=True
    )
    
    status = result.stdout.strip()
    print(f"   Provisioning state: {status}")
    
    if status != "Succeeded":
        print(f"\n⚠️  Cluster not ready yet. Current state: {status}")
        print("   Wait a few more minutes and re-run this cell.")
    else:
        # Get AKS credentials
        print("\n🔐 Downloading AKS credentials...")
        result = subprocess.run(
            [
                "az", "aks", "get-credentials",
                "--resource-group", AZURE_RESOURCE_GROUP,
                "--name", AKS_CLUSTER_NAME,
                "--overwrite-existing"
            ],
            capture_output=True,
            text=True
        )
        
        if result.returncode == 0:
            print("✅ kubectl configured to use AKS cluster")
            
            # Verify connection
            print("\n🔍 Verifying connection:")
            subprocess.run(["kubectl", "cluster-info"])
            
            print("\n🔍 AKS nodes:")
            subprocess.run(["kubectl", "get", "nodes", "-o", "wide"])
            
            print("\n✅ Connected to AKS! Ready to deploy applications.")
        else:
            print(f"❌ Failed to get credentials: {result.stderr}")

## Cell 4 — Deploy Application to AKS

Deploy the same Flask app from the main notebook to Azure. Uses the same YAML manifests.

In [ ]:
# ── Deploy to AKS ──────────────────────────────────────────────────────────

# Create deployment
deployment_yaml = """
apiVersion: apps/v1
kind: Deployment
metadata:
  name: flask-deployment
  labels:
    app: flask
spec:
  replicas: 3
  selector:
    matchLabels:
      app: flask
  template:
    metadata:
      labels:
        app: flask
    spec:
      containers:
      - name: flask
        image: hashicorp/http-echo:latest
        args:
          - "-text=SmartVal API v1.0 (AKS) - Pod: $(HOSTNAME)"
        ports:
        - containerPort: 5678
        env:
        - name: HOSTNAME
          valueFrom:
            fieldRef:
              fieldPath: metadata.name
"""

# Create LoadBalancer service (Azure provides external IP)
service_yaml = """
apiVersion: v1
kind: Service
metadata:
  name: flask-service
spec:
  type: LoadBalancer  # Azure assigns external IP
  selector:
    app: flask
  ports:
  - port: 80
    targetPort: 5678
"""

# Write and apply deployment
deployment_file = "aks-deployment.yaml"
with open(deployment_file, "w") as f:
    f.write(deployment_yaml)

subprocess.run(["kubectl", "apply", "-f", deployment_file])
print("✅ Deployment created (3 replicas)")

# Write and apply service
service_file = "aks-service.yaml"
with open(service_file, "w") as f:
    f.write(service_yaml)

subprocess.run(["kubectl", "apply", "-f", service_file])
print("✅ LoadBalancer service created")

# Wait for pods to start
print("\n⏳ Waiting for pods to become ready...")
time.sleep(10)

# Check deployment status
print("\n🔍 Deployment status:")
subprocess.run(["kubectl", "get", "deployments"])

print("\n🔍 Pods:")
subprocess.run(["kubectl", "get", "pods", "-l", "app=flask", "-o", "wide"])

print("\n🔍 Service (wait for EXTERNAL-IP):")
subprocess.run(["kubectl", "get", "service", "flask-service"])

print("\n💡 Azure LoadBalancer is provisioning an external IP (takes 2-3 minutes).")
print("   Re-run: kubectl get service flask-service")
print("   Once EXTERNAL-IP appears, access your app at: http://<EXTERNAL-IP>")

# Clean up YAML files
for f in [deployment_file, service_file]:
    if os.path.exists(f):
        os.remove(f)

## Cell 5 — Scale Cluster Nodes (Horizontal Scaling)

Add more nodes to the cluster for increased capacity. This is horizontal scaling at the infrastructure level.

In [ ]:
# ── Scale AKS Nodes ────────────────────────────────────────────────────────

NEW_NODE_COUNT = 3  # Scale from 2 to 3 nodes

if not AZURE_SUBSCRIPTION_ID:
    print("❌ Cannot proceed without AZURE_SUBSCRIPTION_ID. See Cell 1.")
else:
    print(f"📈 Scaling AKS cluster to {NEW_NODE_COUNT} nodes...")
    print(f"   💰 Cost will increase to ~${0.10 * NEW_NODE_COUNT:.2f}/hour")
    
    result = subprocess.run(
        [
            "az", "aks", "scale",
            "--resource-group", AZURE_RESOURCE_GROUP,
            "--name", AKS_CLUSTER_NAME,
            "--node-count", str(NEW_NODE_COUNT)
        ],
        capture_output=True,
        text=True
    )
    
    if result.returncode == 0:
        print("✅ Scaling operation started")
        
        # Wait for scale operation
        time.sleep(30)
        
        # Check nodes
        print("\n🔍 Cluster nodes:")
        subprocess.run(["kubectl", "get", "nodes"])
        
        print("\n✅ Cluster scaled! More capacity available for pods.")
    else:
        print(f"❌ Scaling failed: {result.stderr}")

print("\n💡 Scaling nodes = more infrastructure capacity")
print("   Scaling pods (replicas) = more app instances on existing nodes")
print("\n   Scale pods: kubectl scale deployment flask-deployment --replicas=5")

## Cell 6 — Monitor with Azure Monitor for Containers

Enable Azure's monitoring solution for insights into cluster health, pod metrics, and logs.

In [ ]:
# ── Enable Azure Monitor ───────────────────────────────────────────────────

if not AZURE_SUBSCRIPTION_ID:
    print("❌ Cannot proceed without AZURE_SUBSCRIPTION_ID. See Cell 1.")
else:
    # Check if monitoring is enabled
    print("🔍 Checking Azure Monitor status...")
    result = subprocess.run(
        [
            "az", "aks", "show",
            "--resource-group", AZURE_RESOURCE_GROUP,
            "--name", AKS_CLUSTER_NAME,
            "--query", "addonProfiles.omsagent.enabled",
            "-o", "tsv"
        ],
        capture_output=True,
        text=True
    )
    
    if "true" in result.stdout.lower():
        print("✅ Azure Monitor is already enabled")
    else:
        print("📊 Enabling Azure Monitor for Containers...")
        result = subprocess.run(
            [
                "az", "aks", "enable-addons",
                "--resource-group", AZURE_RESOURCE_GROUP,
                "--name", AKS_CLUSTER_NAME,
                "--addons", "monitoring"
            ],
            capture_output=True,
            text=True
        )
        
        if result.returncode == 0:
            print("✅ Azure Monitor enabled")
        else:
            print(f"⚠️  Enable failed: {result.stderr}")
    
    # Get cluster resource ID for portal link
    result = subprocess.run(
        [
            "az", "aks", "show",
            "--resource-group", AZURE_RESOURCE_GROUP,
            "--name", AKS_CLUSTER_NAME,
            "--query", "id",
            "-o", "tsv"
        ],
        capture_output=True,
        text=True
    )
    
    cluster_id = result.stdout.strip()
    
    print("\n📊 View metrics in Azure Portal:")
    print(f"   https://portal.azure.com/#resource{cluster_id}/containerLogs")
    
    print("\n💡 Azure Monitor provides:")
    print("   • CPU/memory usage per pod and node")
    print("   • Container logs (searchable)")
    print("   • Performance metrics (latency, throughput)")
    print("   • Alerts and dashboards")
    
    print("\n🔍 Query logs using:")
    print("   kubectl logs <pod-name>                    # Local")
    print("   az monitor log-analytics query ...         # Azure Log Analytics")

print("\n" + "="*70)
print("🧹 CLEANUP (avoid ongoing charges):")
print("="*70)
print(f"az aks delete --name {AKS_CLUSTER_NAME} --resource-group {AZURE_RESOURCE_GROUP} --yes --no-wait")
print(f"az group delete --name {AZURE_RESOURCE_GROUP} --yes --no-wait")
print("\n⚠️  Run these commands when done to stop incurring costs!")
print("="*70)

## Summary

You've deployed to production-grade Azure Kubernetes Service:

✅ **Created AKS cluster** — managed K8s in Azure  
✅ **Connected kubectl** — same CLI as local Kind  
✅ **Deployed with LoadBalancer** — Azure assigned external IP  
✅ **Scaled cluster nodes** — horizontal infrastructure scaling  
✅ **Enabled Azure Monitor** — metrics, logs, and alerts

**Key differences from Kind (local):**
- **LoadBalancer service type** — Azure provisions real external IPs (Kind uses NodePort)
- **Multi-node by default** — production resilience
- **Managed control plane** — Azure handles master nodes
- **Cloud integration** — LoadBalancers, managed disks, Azure AD

**Cost management:**
- Always delete clusters after testing: `az aks delete ...`
- Use B-series VMs for low-cost demos
- Enable autoscaling to reduce idle costs
- Monitor spending: `az consumption usage list`

**Next:** Ch.4 (CI/CD Pipelines) automates deployments to AKS from GitHub Actions.